# Stanford RNA 3D Folding Part 2 - TRM Submission

This notebook uses Tiny Recursive Models (TRM) to predict RNA 3D structures.

## Overview
- Predicts 5 different conformations per RNA sequence
- Uses recursive reasoning for complex RNA folding patterns
- Outputs C1' atom coordinates in competition format

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from typing import List, Dict
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Model Definition

In [ ]:
class RNAStructureModel(nn.Module):
    """TRM-based model for RNA 3D structure prediction."""
    
    def __init__(
        self,
        vocab_size: int = 4,
        embed_dim: int = 256,
        hidden_dim: int = 512,
        num_structures: int = 5,
        max_length: int = 500,
        H_cycles: int = 3,
        L_cycles: int = 6,
        L_layers: int = 2,
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.num_structures = num_structures
        self.max_length = max_length
        
        # Embeddings
        self.nucleotide_embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=vocab_size)
        self.position_embedding = nn.Embedding(max_length, embed_dim)
        
        # Transformer layers
        self.reasoning_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=8,
                dim_feedforward=hidden_dim,
                dropout=0.1,
                batch_first=True
            )
            for _ in range(L_layers)
        ])
        
        # Output heads
        self.structure_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 3)
            )
            for _ in range(num_structures)
        ])
        
        self.H_cycles = H_cycles
        self.L_cycles = L_cycles
    
    def forward(self, sequences: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = sequences.shape
        
        # Embed and add positions
        sequences_clamped = torch.clamp(sequences, min=0)
        x = self.nucleotide_embedding(sequences_clamped)
        positions = torch.arange(seq_len, device=sequences.device).unsqueeze(0)
        x = x + self.position_embedding(positions)
        
        # Attention mask
        padding_mask = sequences == self.vocab_size
        
        # Recursive reasoning
        for h_cycle in range(self.H_cycles):
            for l_cycle in range(self.L_cycles):
                for layer in self.reasoning_layers:
                    x = layer(x, src_key_padding_mask=padding_mask)
        
        # Predict structures
        predictions = [head(x) for head in self.structure_heads]
        predictions = torch.stack(predictions, dim=2)
        
        return predictions

print("Model class defined")

## Helper Functions

In [ ]:
def encode_rna_sequence(sequence: str, max_length: int = 500) -> np.ndarray:
    """Encode RNA sequence to numerical format."""
    nucleotide_map = {'A': 0, 'C': 1, 'G': 2, 'U': 3, 'N': 4}
    encoded = np.array([nucleotide_map.get(n, 4) for n in sequence.upper()])
    
    if len(encoded) < max_length:
        encoded = np.pad(encoded, (0, max_length - len(encoded)), constant_values=4)
    else:
        encoded = encoded[:max_length]
    
    return encoded

print("Helper functions defined")

## Load Test Data

In [ ]:
# Load test sequences
test_df = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv')
print(f"Loaded {len(test_df)} test sequences")
print(f"\nColumns: {test_df.columns.tolist()}")
print(f"\nFirst sequence:")
print(test_df.head(1))

## Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model configuration
max_length = 500
batch_size = 32 if torch.cuda.is_available() else 8

model = RNAStructureModel(
    vocab_size=4,
    embed_dim=256,
    hidden_dim=512,
    num_structures=5,
    max_length=max_length,
    H_cycles=3,
    L_cycles=6,
    L_layers=2,
).to(device)

# Load pretrained weights if available
checkpoint_path = '/kaggle/input/rna-structure-weights/rna_model.pth'
if os.path.exists(checkpoint_path):
    print(f"Loading weights from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print("Weights loaded successfully")
else:
    print("WARNING: No pretrained weights found - using random initialization")

model.eval()
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

## Generate Predictions

In [ ]:
predictions = []

with torch.no_grad():
    for i in tqdm(range(0, len(test_df), batch_size), desc="Predicting"):
        batch_df = test_df.iloc[i:i + batch_size]
        
        # Encode sequences
        encoded_seqs = []
        for _, row in batch_df.iterrows():
            encoded = encode_rna_sequence(row['sequence'], max_length)
            encoded_seqs.append(torch.tensor(encoded, dtype=torch.long))
        
        # Stack and predict
        encoded_batch = torch.stack(encoded_seqs).to(device)
        pred_coords = model(encoded_batch)
        
        # Clip coordinates to valid range
        pred_coords = torch.clamp(pred_coords, -999.999, 9999.999)
        pred_coords = pred_coords.cpu().numpy()
        
        # Store predictions
        for j, (idx, row) in enumerate(batch_df.iterrows()):
            seq_len = len(row['sequence'])
            predictions.append({
                'target_id': row['target_id'],
                'sequence': row['sequence'],
                'coordinates': pred_coords[j, :seq_len, :, :]
            })

print(f"\nGenerated predictions for {len(predictions)} sequences")

## Create Submission File

In [ ]:
submission_rows = []

for pred in predictions:
    target_id = pred['target_id']
    sequence = pred['sequence']
    coords = pred['coordinates']  # (seq_len, 5, 3)
    
    for i, nucleotide in enumerate(sequence):
        row = {
            'ID': f"{target_id}_{i + 1}",
            'resname': nucleotide,
            'resid': i + 1,
        }
        
        # Add coordinates for 5 structures
        for struct_idx in range(5):
            x, y, z = coords[i, struct_idx, :]
            row[f'x_{struct_idx + 1}'] = x
            row[f'y_{struct_idx + 1}'] = y
            row[f'z_{struct_idx + 1}'] = z
        
        submission_rows.append(row)

# Create DataFrame with correct column order
submission_df = pd.DataFrame(submission_rows)
coord_cols = []
for i in range(1, 6):
    coord_cols.extend([f'x_{i}', f'y_{i}', f'z_{i}'])
columns = ['ID', 'resname', 'resid'] + coord_cols
submission_df = submission_df[columns]

# Save submission
submission_df.to_csv('submission.csv', index=False)

print(f"Submission file created!")
print(f"Total residues: {len(submission_df)}")
print(f"From {len(predictions)} sequences")
print(f"\nFirst few rows:")
print(submission_df.head())

## Verify Submission Format

In [ ]:
# Verify format
print("Submission verification:")
print(f"  Shape: {submission_df.shape}")
print(f"  Columns: {len(submission_df.columns)}")
print(f"  Expected columns: 18 (ID, resname, resid + 15 coordinates)")
print(f"  Column check: {'✓' if len(submission_df.columns) == 18 else '✗'}")
print(f"\n  Coordinate ranges:")
for i in range(1, 6):
    x_col = f'x_{i}'
    if x_col in submission_df.columns:
        x_min = submission_df[x_col].min()
        x_max = submission_df[x_col].max()
        print(f"    Structure {i}: x ∈ [{x_min:.3f}, {x_max:.3f}]")

print("\n✓ Submission ready!")

## RNA Training Script - Imports and Setup

In [ ]:
#!/usr/bin/env python3
"""
RNA 3D Structure Prediction Training Script

This script trains a Tiny Recursive Model (TRM) to predict 3D RNA structures.
The model learns to predict C1' atom coordinates for each nucleotide in an RNA sequence,
generating 5 different conformations as required by the competition format.
"""

import os
import argparse
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm import tqdm

# Import TRM components
# Note: TinyRecursionModel import removed - using transformer-based implementation
# Note: get_activation import removed - using ReLU activation


## RNA Dataset Class

In [ ]:
class RNAStructureDataset(Dataset):
    """Dataset for RNA 3D structure prediction."""
    
    def __init__(self, sequences_file: str, labels_file: str, max_length: int = 500):
        """
        Initialize RNA structure dataset.
        
        Args:
            sequences_file: Path to sequences JSON file
            labels_file: Path to labels JSON file
            max_length: Maximum sequence length
        """
        with open(sequences_file, 'r') as f:
            self.sequences = json.load(f)
        
        with open(labels_file, 'r') as f:
            self.labels = json.load(f)
        
        # Create mapping from target_id to labels
        self.label_map = {label['target_id']: label for label in self.labels}
        
        # Filter sequences that have labels
        self.sequences = [seq for seq in self.sequences 
                         if seq['target_id'] in self.label_map]
        
        self.max_length = max_length
    
    def __len__(self) -> int:
        return len(self.sequences)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get a sample from the dataset.
        
        Returns:
            Tuple of (encoded_sequence, coordinates)
            - encoded_sequence: (max_length,) tensor with nucleotide encoding
            - coordinates: (seq_len, num_structures, 3) tensor with 3D coordinates
        """
        seq_data = self.sequences[idx]
        label_data = self.label_map[seq_data['target_id']]
        
        # Get encoded sequence
        encoded_seq = torch.tensor(seq_data['encoded_sequence'], dtype=torch.long)
        
        # Get coordinates
        coords_list = []
        for residue in label_data['residues']:
            coords = torch.tensor(residue['coordinates'], dtype=torch.float32)
            coords_list.append(coords)
        
        # Stack coordinates: (seq_len, num_structures, 3)
        if coords_list:
            coordinates = torch.stack(coords_list)
        else:
            # Empty coordinates
            coordinates = torch.zeros((self.max_length, 1, 3), dtype=torch.float32)
        
        # Pad coordinates to max_length
        if coordinates.shape[0] < self.max_length:
            padding = torch.zeros(
                (self.max_length - coordinates.shape[0], coordinates.shape[1], 3),
                dtype=torch.float32
            )
            coordinates = torch.cat([coordinates, padding], dim=0)
        else:
            coordinates = coordinates[:self.max_length]
        
        return encoded_seq, coordinates

## RNA Structure Model Architecture

In [ ]:
class RNAStructureModel(nn.Module):
    """TRM-based model for RNA 3D structure prediction."""
    
    def __init__(
        self,
        vocab_size: int = 4,  # A, C, G, U
        embed_dim: int = 256,
        hidden_dim: int = 512,
        num_structures: int = 5,
        max_length: int = 500,
        H_cycles: int = 3,
        L_cycles: int = 6,
        L_layers: int = 2,
    ):
        """
        Initialize RNA structure prediction model.
        
        Args:
            vocab_size: Number of nucleotide types
            embed_dim: Embedding dimension
            hidden_dim: Hidden dimension for TRM
            num_structures: Number of structures to predict
            max_length: Maximum sequence length
            H_cycles: Number of high-level reasoning cycles
            L_cycles: Number of low-level reasoning cycles
            L_layers: Number of layers in TRM
        """
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.num_structures = num_structures
        self.max_length = max_length
        
        # Nucleotide embedding (vocab_size=4 for ACGU, +1 for padding)
        self.nucleotide_embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=vocab_size)
        
        # Position embedding
        self.position_embedding = nn.Embedding(max_length, embed_dim)
        
        # TRM backbone (we'll use a simplified version)
        self.reasoning_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=8,
                dim_feedforward=hidden_dim,
                dropout=0.1,
                batch_first=True
            )
            for _ in range(L_layers)
        ])
        
        # Output heads for each structure
        self.structure_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 3)  # x, y, z coordinates
            )
            for _ in range(num_structures)
        ])
        
        # Recursive refinement
        self.H_cycles = H_cycles
        self.L_cycles = L_cycles
    
    def forward(self, sequences: torch.Tensor) -> torch.Tensor:
        """
        Forward pass to predict 3D structures.
        
        Args:
            sequences: (batch_size, max_length) tensor of encoded sequences
            
        Returns:
            predictions: (batch_size, max_length, num_structures, 3) tensor
        """
        batch_size, seq_len = sequences.shape
        
        # Embed nucleotides
        # Handle padding index 4 by clamping to valid range
        sequences_clamped = torch.clamp(sequences, min=0)
        x = self.nucleotide_embedding(sequences_clamped)  # (batch, seq_len, embed_dim)
        
        # Add position embeddings
        positions = torch.arange(seq_len, device=sequences.device).unsqueeze(0)
        x = x + self.position_embedding(positions)
        
        # Create attention mask for padding (padding index is vocab_size, which is 4)
        padding_mask = sequences == self.vocab_size  # (batch, seq_len)
        
        # Recursive reasoning with multiple cycles
        for h_cycle in range(self.H_cycles):
            for l_cycle in range(self.L_cycles):
                # Apply transformer layers
                for layer in self.reasoning_layers:
                    x = layer(x, src_key_padding_mask=padding_mask)
        
        # Predict structures with each head
        predictions = []
        for head in self.structure_heads:
            coords = head(x)  # (batch, seq_len, 3)
            predictions.append(coords)
        
        # Stack predictions: (batch, seq_len, num_structures, 3)
        predictions = torch.stack(predictions, dim=2)
        
        return predictions

## Training Functions

In [ ]:
def train_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: optim.Optimizer,
    device: torch.device,
    clip_coords: bool = True
) -> float:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    for sequences, targets in tqdm(dataloader, desc="Training"):
        sequences = sequences.to(device)
        targets = targets.to(device)
        
        # Forward pass
        predictions = model(sequences)
        
        # Clip coordinates if needed (competition requirement)
        if clip_coords:
            predictions = torch.clamp(predictions, -999.999, 9999.999)
        
        # Compute loss (MSE on coordinates)
        # Only compute loss on non-padding positions (padding index is 4)
        mask = (sequences != 4).unsqueeze(-1).unsqueeze(-1)  # (batch, seq_len, 1, 1)
        
        # If targets have fewer structures, repeat the last one
        if targets.shape[2] < predictions.shape[2]:
            last_structure = targets[:, :, -1:, :]  # (batch, seq_len, 1, 3)
            padding_structures = last_structure.repeat(
                1, 1, predictions.shape[2] - targets.shape[2], 1
            )
            targets = torch.cat([targets, padding_structures], dim=2)
        
        loss = ((predictions - targets) ** 2 * mask).sum() / mask.sum()
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

## Validation Function

In [ ]:
def validate(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device
) -> float:
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for sequences, targets in tqdm(dataloader, desc="Validating"):
            sequences = sequences.to(device)
            targets = targets.to(device)
            
            # Forward pass
            predictions = model(sequences)
            predictions = torch.clamp(predictions, -999.999, 9999.999)
            
            # Compute loss
            mask = (sequences != 4).unsqueeze(-1).unsqueeze(-1)
            
            # Handle different number of structures
            if targets.shape[2] < predictions.shape[2]:
                last_structure = targets[:, :, -1:, :]
                padding_structures = last_structure.repeat(
                    1, 1, predictions.shape[2] - targets.shape[2], 1
                )
                targets = torch.cat([targets, padding_structures], dim=2)
            
            loss = ((predictions - targets) ** 2 * mask).sum() / mask.sum()
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

## Main Training Loop

**Note**: This function uses argparse for command-line arguments. In a notebook environment, you can either:
1. Set parameters directly as variables before calling functions
2. Use the function with default arguments
3. Modify the code to use notebook-friendly parameter passing

In [ ]:
def main():
    parser = argparse.ArgumentParser(description='Train TRM for RNA 3D structure prediction')
    parser.add_argument('--data-dir', type=str, default='data/rna-structure',
                        help='Directory with processed RNA data')
    parser.add_argument('--output-dir', type=str, default='checkpoints/rna',
                        help='Directory to save model checkpoints')
    parser.add_argument('--batch-size', type=int, default=16,
                        help='Batch size for training')
    parser.add_argument('--epochs', type=int, default=100,
                        help='Number of training epochs')
    parser.add_argument('--lr', type=float, default=1e-4,
                        help='Learning rate')
    parser.add_argument('--max-length', type=int, default=500,
                        help='Maximum sequence length')
    parser.add_argument('--num-structures', type=int, default=5,
                        help='Number of structures to predict')
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu',
                        help='Device to use for training')
    
    args = parser.parse_args()
    
    # Create output directory
    output_path = Path(args.output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Load datasets
    print("Loading datasets...")
    train_dataset = RNAStructureDataset(
        f"{args.data_dir}/train_sequences.json",
        f"{args.data_dir}/train_labels.json",
        args.max_length
    )
    
    val_dataset = RNAStructureDataset(
        f"{args.data_dir}/val_sequences.json",
        f"{args.data_dir}/val_labels.json",
        args.max_length
    )
    
    print(f"Train dataset: {len(train_dataset)} samples")
    print(f"Val dataset: {len(val_dataset)} samples")
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=4
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=4
    )
    
    # Create model
    print("Creating model...")
    device = torch.device(args.device)
    model = RNAStructureModel(
        num_structures=args.num_structures,
        max_length=args.max_length
    ).to(device)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Create optimizer
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    
    # Training loop
    best_val_loss = float('inf')
    
    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch + 1}/{args.epochs}")
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, device)
        print(f"Train loss: {train_loss:.6f}")
        
        # Validate
        val_loss = validate(model, val_loader, device)
        print(f"Val loss: {val_loss:.6f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, output_path / 'best_model.pth')
            print(f"Saved best model (val_loss: {val_loss:.6f})")
        
        # Save checkpoint
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, output_path / f'checkpoint_epoch_{epoch + 1}.pth')
    
    print("\nTraining complete!")
    print(f"Best validation loss: {best_val_loss:.6f}")

## Script Entry Point

**Note**: The `if __name__ == '__main__':` pattern is used for standalone scripts. In a notebook, you can directly call functions or set up training with custom parameters in separate cells.

In [ ]:
if __name__ == '__main__':
    main()